# ReqSeek — Reproducible Public Dataset Release

This notebook is located in the public `datasets/` directory. It **reads the complete private copies** of ARID and SwaRD from `../datasets_with_data/`, merges ARID's train/test splits, removes only the source groups selected below, and writes the public-release copies to the current `datasets/` directory.

- Change an entry in `EXCLUDE_SOURCES` from `True` to `False` when redistribution permission is confirmed; then rerun all cells.
- The original `../datasets_with_data/ARID/` and `../datasets_with_data/SwaRD/` remain unchanged.
- The published metadata is derived from the full source metadata and excludes source attributions and bibliography entries for withheld sources; **reference numbers are deliberately not renumbered**.
- Review the printed before/after tables before writing the release files.
- **Reproducibility note:** The notebook documents the release-generation process, but reproducing it requires access to the complete source datasets, which are not included in the public repository.
- **Git caution:** A clean current dataset does **not** remove records from older public Git commits or archived versions.

In [1]:
import datasets
import numpy as np
import pandas as pd

In [2]:
sward = datasets.load_from_disk('../datasets_with_data/SwaRD/')
sward = sward.rename_column('isF/NF', 'F/NF Classes')
sward = sward.map(lambda row: {"F/NF Classes": None})
sward = sward.map(lambda row: {"NF Subclasses": None})

In [3]:
arid = datasets.load_from_disk('../datasets_with_data/ARID/')
arid_merged = datasets.concatenate_datasets([arid['train'], arid['test']])

In [4]:
# Copy the complete original metadata from the loaded datasets.
import copy

ARID_ORIGINAL_METADATA = copy.deepcopy(vars(arid_merged.info))
SWARD_ORIGINAL_METADATA = copy.deepcopy(vars(sward.info))

assert ARID_ORIGINAL_METADATA.get('license'), 'ARID source .info.license is missing'
assert SWARD_ORIGINAL_METADATA.get('license'), 'SwaRD source .info.license is missing'

print('Original ARID and SwaRD licensing metadata copied from their .info objects.')

Original ARID and SwaRD licensing metadata copied from their .info objects.


## Configure which source groups are withheld

Set a value to `False` once redistribution permission for that group is confirmed. Each switch controls both the data rows **and** the corresponding licensing/reference entries. The 13 COEST subprojects are handled together through `Project Name == "COEST"`.

In [5]:
from pathlib import Path
from collections import Counter
import copy
import json
import re
import shutil
import uuid

# False = INCLUDE in public release; True = WITHHOLD from public release.
EXCLUDE_SOURCES = {
    "Smarthome": True,
    "Dronology": True,
    "OPENCOSS": True,
    "COEST": True,  # all 13 COEST constituent projects
}

PROJECT_NAMES = {
    "Smarthome": "A Dataset of Crowdsourced Smarthome Requirements with Creativity Ratings",
    "Dronology": "Dronology",
    "OPENCOSS": "OPENCOSS",
    "COEST": "COEST",
}

# Set True only AFTER inspecting reports; removes/replaces ONLY ../datasets/ARID and ../datasets/SwaRD.
# The full inputs ('../datasets_with_data/ARID/' and '../datasets_with_data/SwaRD/') are never overwritten.
OVERWRITE_RELEASE_OUTPUTS = False
RELEASE_ROOT = Path('./')

# These originals are included in the notebook so the processing is repeatable
# without relying on external JSON files. They are not edited in-place.

for key, project in PROJECT_NAMES.items():
    for dataset_name, dataset in [("ARID", arid_merged), ("SwaRD", sward)]:
        observed = set(dataset.unique('Project Name'))
        if project not in observed:
            raise ValueError(f"Unexpected project mapping: {key} -> {project!r} missing from {dataset_name}")
            
print("Configured source groups:", EXCLUDE_SOURCES)

Configured source groups: {'Smarthome': True, 'Dronology': True, 'OPENCOSS': True, 'COEST': True}


## Filter both datasets and calculate counts by class

The code checks all four project names exist, counts removed and retained rows independently, and preserves each dataset's original column types. No class values are changed.

In [6]:
# Classes used by the paper. Unknown labels are still counted under their exact raw spelling.
def class_report(dataset, class_column, dataset_name):
    values = pd.Series(dataset[class_column]).fillna('<missing>').astype(str)
    return values.value_counts(dropna=False).rename_axis('class_label').reset_index(name = 'count')

# Only filter the merged ARID (not its individual train/test splits).
def group_rows(dataset, group):
    return dataset.filter(lambda row: row['Project Name'] == PROJECT_NAMES[group])

def build_release(dataset, dataset_name, class_column):
    original_n = len(dataset)
    withholding = [g for g, flag in EXCLUDE_SOURCES.items() if flag]
    excluded_project_names = {PROJECT_NAMES[g] for g in withholding}
    kept = dataset.filter(lambda row: row['Project Name'] not in excluded_project_names)
    removed = dataset.filter(lambda row: row['Project Name'] in excluded_project_names)
    assert original_n == len(kept) + len(removed)
    assert excluded_project_names.isdisjoint(set(kept.unique('Project Name')))
    rows = []
    for group in PROJECT_NAMES:
        sub = group_rows(dataset, group)
        labels = pd.Series(sub[class_column]).fillna('<missing>').astype(str).value_counts()
        record = {'dataset': dataset_name, 'source_group': group,
                  'status': 'excluded' if EXCLUDE_SOURCES[group] else 'included',
                  'total_records': len(sub)}
        record.update({str(label): int(n) for label, n in labels.items()})
        rows.append(record)
    report = pd.DataFrame(rows).fillna(0)
    return kept, removed, report

arid_release, arid_withheld, arid_report = build_release(arid_merged, 'ARID', 'ternary_cls')
sward_release, sward_withheld, sward_report = build_release(sward, 'SwaRD', 'isReqSysAuxContAux')
release_report = pd.concat([arid_report, sward_report], ignore_index=True)

print('Original / withheld / public records:')
display(pd.DataFrame([{'dataset':'ARID', 'full':len(arid_merged), 'withheld':len(arid_withheld), 'public':len(arid_release)},
                      {'dataset':'SwaRD','full':len(sward), 'withheld':len(sward_withheld),'public':len(sward_release)}]))

print('Detailed source totals by dataset and original class label:')
display(release_report)

# Optional drill-down: all COEST subprojects, identified from the actual Subproject Name column.
coest_subproject_reports = []

for name, dataset, class_col in [('ARID', arid_merged, 'ternary_cls'), ('SwaRD', sward, 'isReqSysAuxContAux')]:
    coest = group_rows(dataset, 'COEST')
    if len(coest):
        frame = pd.DataFrame({
            'dataset': [name] * len(coest),
            'subproject': pd.Series(coest['Subproject Name']).fillna('<missing>').astype(str),
            'class_label': pd.Series(coest[class_col]).fillna('<missing>').astype(str)})
        
        grouped = frame.groupby(['dataset','subproject','class_label'], dropna=False).size().unstack(fill_value=0).reset_index()
        counts = frame.groupby(['dataset','subproject'],dropna=False).size().rename('total_records').reset_index()
        grouped = grouped.merge(counts,on=['dataset','subproject'],how='left')
        grouped['status'] = 'excluded' if EXCLUDE_SOURCES['COEST'] else 'included'
        coest_subproject_reports.append(grouped)
        
coest_subproject_report = pd.concat(coest_subproject_reports, ignore_index=True) if coest_subproject_reports else pd.DataFrame()

print('COEST subproject detail:')
display(coest_subproject_report)
print('Full class distributions:')
display(class_report(arid_merged, 'ternary_cls', 'ARID').assign(dataset='ARID'))
display(class_report(sward, 'isReqSysAuxContAux', 'SwaRD').assign(dataset='SwaRD'))
print('Withheld class distributions:')
display(class_report(arid_withheld, 'ternary_cls', 'ARID').assign(dataset='ARID'))
display(class_report(sward_withheld, 'isReqSysAuxContAux', 'SwaRD').assign(dataset='SwaRD'))
print('Public class distributions:')
display(class_report(arid_release, 'ternary_cls', 'ARID').assign(dataset='ARID'))
display(class_report(sward_release, 'isReqSysAuxContAux', 'SwaRD').assign(dataset='SwaRD'))

# Informational comparison to the counts
if all(EXCLUDE_SOURCES.values()) and len(sward_withheld) != 7262:
    print(f'NOTE: SwaRD excluded count = {len(sward_withheld)}, not previously discussed 7262. Inspect the row-by-row report before release.')

Original / withheld / public records:


,dataset,full,withheld,public
0,ARID,1609,278,1331
1,SwaRD,40522,7262,33260


Detailed source totals by dataset and original class label:


,dataset,source_group,status,total_records,contextual_auxiliary,requirement,system_related_auxiliary
0,ARID,Smarthome,excluded,22,22.0,0.0,0.0
1,ARID,Dronology,excluded,23,6.0,13.0,4.0
2,ARID,OPENCOSS,excluded,11,0.0,10.0,1.0
3,ARID,COEST,excluded,222,39.0,147.0,36.0
4,SwaRD,Smarthome,excluded,2902,1471.0,507.0,924.0
5,SwaRD,Dronology,excluded,618,25.0,229.0,364.0
6,SwaRD,OPENCOSS,excluded,109,0.0,101.0,8.0
7,SwaRD,COEST,excluded,3633,49.0,2196.0,1388.0


COEST subproject detail:


,dataset,subproject,contextual_auxiliary,requirement,system_related_auxiliary,total_records,status
0,ARID,CCHIT,0,3,0,3,excluded
1,ARID,CM1-NASA,0,4,3,7,excluded
2,ARID,EBT,0,0,1,1,excluded
3,ARID,GANNT,1,5,1,7,excluded
4,ARID,HIPAA,28,101,26,155,excluded
5,ARID,InfusionPump,1,8,1,10,excluded
6,ARID,Kiosk,3,16,1,20,excluded
7,ARID,SMOS,1,0,0,1,excluded
8,ARID,WARC,0,10,1,11,excluded
9,ARID,eANCI,1,0,0,1,excluded


... (5 outputs left out)

,class_label,count,dataset
0,requirement,3033,SwaRD
1,system_related_auxiliary,2684,SwaRD
2,contextual_auxiliary,1545,SwaRD


Public class distributions:


,class_label,count,dataset
0,contextual_auxiliary,531,ARID
1,requirement,516,ARID
2,system_related_auxiliary,284,ARID


,class_label,count,dataset
0,system_related_auxiliary,23062,SwaRD
1,requirement,7168,SwaRD
2,contextual_auxiliary,3030,SwaRD


## Rebuild source-specific licensing and citations

The full uploaded metadata is the **single source of truth**. Each rerun starts from its original license text; it does not repeatedly delete or modify an already-filtered license. Entries and their bibliography references are removed together. Original citation numbers are intentionally retained (gaps are expected). A brief release-scope notice is added.

In [7]:
# Bibliography IDs are intentionally *not* renumbered when excluded references are removed.
REFERENCE_IDS = {'ARID': {'Smarthome': {14,15}, 'Dronology': {12,13}, 'OPENCOSS': {23,24}, 'COEST': set(range(25,36))},
                 'SwaRD': {'Smarthome': {16,17}, 'Dronology': {14,15}, 'OPENCOSS': {24,25}, 'COEST': set(range(26,37))}}

# The original metadata was supplied by the dataset authors. The code removes
# attribution entries ONLY for currently excluded sources and leaves all others verbatim.
def make_release_license(original_text, kind):
    excluded_ids = set().union(*(REFERENCE_IDS[kind][g] for g in EXCLUDE_SOURCES if EXCLUDE_SOURCES[g])) if any(EXCLUDE_SOURCES.values()) else set()
    excluded_names = {g for g, flag in EXCLUDE_SOURCES.items() if flag}
    lines = original_text.splitlines(keepends = True)
    excluded_lines = set()
    in_coest = False
    in_references = False
    drop_reference = False

    for i, line in enumerate(lines):
        heading = re.match(r'^\s*#{2,3}\s+\*\*(.*?)\*\*\s*$', line.strip())
        if heading:
            name = heading.group(1)
            in_coest = name == 'COEST Community Dataset'
            in_references = name == 'References'
            drop_reference = False
            if in_coest and 'COEST' in excluded_names:
                excluded_lines.add(i)
            continue
        if in_references:
            m = re.match(r'^\s*\[(\d+)\]\s', line)
            if m:
                drop_reference = int(m.group(1)) in excluded_ids
            if drop_reference:
                excluded_lines.add(i)
            continue
        if in_coest and 'COEST' in excluded_names:
            excluded_lines.add(i)
            continue
        for name, marker in [('Dronology', 'Dronology'), ('Smarthome', 'Smarthome Requirements Dataset'),('OPENCOSS', 'OPENCOSS')]:
            if name in excluded_names and re.search(r'^\s*•\s*\*\*'+re.escape(marker)+r'\*\*', line):
                excluded_lines.add(i)

    license_text = ''.join(line for i,line in enumerate(lines) if i not in excluded_lines)

    # Remove empty license categories. Preserve the CC BY 4.0 / CC BY-SA / WHO / etc.
    for title in ('No Official License --- Citation Requested', 'GNU General Public License v3.0 (GPL-3.0)'):
        license_text = re.sub(r'(?m)^#{2,3}\s+\*\*' + re.escape(title) + r'\*\*\s*\n(?=\s*#{2,3}\s+\*\*)', '', license_text)
    license_text = re.sub(r'\n{4,}', '\n\n\n', license_text)

    if excluded_names:
        scope_note = ('**Public release scope:** This release excludes selected source datasets while redistribution permissions '
                      'are clarified. The original research dataset used for the associated paper may contain additional records. '
                      'The source-specific attribution list below applies to the included sources.')
        marker = re.search(r'(?m)^#{1,3}\s+\*\*Dataset Attributions by License\*\*', license_text)
        if not marker:
            raise AssertionError(f'Attributions heading not found in {kind}')
        license_text = license_text[:marker.start()] + scope_note + '\n\n' + license_text[marker.start():]

    # Validate references exactly match expected IDs with intentional gaps.
    original_refs = set(map(int, re.findall(r'(?m)^\[(\d+)\]\s', original_text.split('## **References**',1)[1])))
    included_refs = set(map(int, re.findall(r'(?m)^\[(\d+)\]\s', license_text.split('## **References**',1)[1])))
    assert included_refs == original_refs - excluded_ids, (kind, sorted(included_refs ^ (original_refs-excluded_ids)))
    prior_to_refs = license_text.split('## **References**',1)[0]
    for cited in re.findall(r'(?<!\w)\[(\d+)\]', prior_to_refs):
        assert int(cited) in included_refs, (kind, 'dangling reference', cited)
    for excluded in excluded_names:
        if excluded in ('Smarthome','Dronology','OPENCOSS'):
            name = 'Smarthome Requirements Dataset' if excluded == 'Smarthome' else excluded
            assert f'**{name}**' not in prior_to_refs
    if 'COEST' in excluded_names:
        assert 'COEST Community Dataset' not in prior_to_refs
    return license_text


def prepare_metadata(released_dataset, original_metadata, kind, withheld_count):
    meta = copy.deepcopy(original_metadata)
    meta['license'] = make_release_license(meta['license'], kind)
    if withheld_count:
        meta['description'] = meta['description'].rstrip() + (
            '\n\n**Release scope:** This public version excludes selected sources pending redistribution permission. '
            'The associated paper reports results from the complete research dataset; thus its counts may differ from this release.\n')
    expected_cols = list(meta['features'])
    assert expected_cols == released_dataset.column_names, (kind, 'feature/column mismatch',expected_cols,released_dataset.column_names)
    # Preserve the original disk dataset's feature definitions and metadata;
    # only release description/license are replaced.
    released_dataset.info.description = meta['description']
    released_dataset.info.license = meta['license']
    released_dataset.info.citation = meta['citation']
    released_dataset.info.homepage = meta['homepage']
    return meta

arid_release_metadata = prepare_metadata(arid_release, ARID_ORIGINAL_METADATA, 'ARID', len(arid_withheld))
sward_release_metadata = prepare_metadata(sward_release, SWARD_ORIGINAL_METADATA, 'SwaRD', len(sward_withheld))

# Show exact attribution categories; check that the excluded names are no longer listed.
for kind,meta in [('ARID',arid_release_metadata), ('SwaRD',sward_release_metadata)]:
    print('\n',kind,'PUBLIC LICENSE HEADINGS:')
    for line in meta['license'].splitlines():
        if re.match(r'^#{2,3}\s',line):print(line)
    print('Retained reference numbers:', re.findall(r'(?m)^\[(\d+)\]',meta['license']))


 ARID PUBLIC LICENSE HEADINGS:
## **License and Usage Notice**
## **Important Notice to Users:**
## **Creative Commons Attribution 4.0 International (CC BY 4.0)**
## **Creative Commons Attribution-ShareAlike 3.0. (CC-BY-SA-3.0)**
## **Creative Commons Zero v1.0 Universal (CC0 1.0)**
## **Eclipse Public License 2.0 (EPL-2.0)**
## **References**
Retained reference numbers: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '16', '17', '18', '19', '20', '21', '22']

 SwaRD PUBLIC LICENSE HEADINGS:
## **License and Usage Notice**
## **Important Notice to Users:**
## **Dataset Attributions by License**
### **Creative Commons Attribution 4.0 International (CC BY 4.0)**
### **Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO (CC BY-NC-SA 3.0)**
### **Creative Commons Attribution-ShareAlike 3.0 (CC BY-SA 3.0)**
### **Creative Commons Zero v1.0 Universal (CC0 1.0)**
### **Eclipse Public License 2.0 (EPL-2.0)**
## **References**
Retained reference numbers: ['1', '2', '3', '4'

## Optional overlap check on the public subsets

Text overlap is calculated after filtering. This is purely diagnostic.

In [8]:
arid_texts = set(arid_release['Requirement Sentences'])
sward_texts = set(sward_release['Requirement Sentences'])
overlap = arid_texts & sward_texts
print(f'ARID public rows: {len(arid_release)}')
print(f'SwaRD public rows: {len(sward_release)}')
print(f'Overlapping unique sentence texts: {len(overlap)}')
print(f'ARID overlap: {100*len(overlap)/len(arid_texts):.2f}%' if arid_texts else 'ARID empty')
print(f'SwaRD overlap: {100*len(overlap)/len(sward_texts):.2f}%' if sward_texts else 'SwaRD empty')

ARID public rows: 1331
SwaRD public rows: 33260
Overlapping unique sentence texts: 428
ARID overlap: 35.37%
SwaRD overlap: 1.32%


## Save the checked public release

**Run only when satisfied with the report above.** To avoid accidentally overwriting existing published outputs, the notebook defaults to `OVERWRITE_RELEASE_OUTPUTS = False`.

Set it to `True` above to replace only `./ARID/` and `./SwaRD/` within the public `datasets/` directory. The complete private datasets in `../datasets_with_data/` remain unchanged.

Report files will be saved in the current `datasets/` directory.

**Git caution:** Replacing the public dataset files does not remove previously committed data from Git history or existing archives.

In [9]:
def write_release():
    RELEASE_ROOT.mkdir(parents=True, exist_ok = True)
    targets = [RELEASE_ROOT/'ARID', RELEASE_ROOT/'SwaRD']
    existing = [str(t) for t in targets if t.exists()]
    if existing and not OVERWRITE_RELEASE_OUTPUTS:
        raise FileExistsError(
            f'Public release output already exists: {existing}. Inspect the reports; '
            'set OVERWRITE_RELEASE_OUTPUTS=True when you intentionally want to replace it.')

    # Stage both outputs before replacing either destination.
    staging = [RELEASE_ROOT/f'.staging_{name}_{uuid.uuid4().hex[:12]}' for name in ('ARID','SwaRD')]
    try:
        arid_release.save_to_disk(str(staging[0]))
        sward_release.save_to_disk(str(staging[1]))
        for stage, target in zip(staging, targets):
            if target.exists():
                if not OVERWRITE_RELEASE_OUTPUTS:
                    raise FileExistsError(str(target))
                shutil.rmtree(target)
            stage.rename(target)
    finally:
        for stage in staging:
            if stage.exists():
                shutil.rmtree(stage)

    # Save generated reports in a separate directory.
#     REPORTS_ROOT = RELEASE_ROOT / 'release_reports'
#     REPORTS_ROOT.mkdir(parents = True, exist_ok = True)
    
#     release_report.to_csv(REPORTS_ROOT / 'source_release_counts.csv', index = False)
#     coest_subproject_report.to_csv(REPORTS_ROOT / 'coest_subproject_counts.csv', index = False)

#     totals = {'ARID': {'full': len(arid_merged), 'excluded': len(arid_withheld), 'released': len(arid_release)},
#               'SwaRD': {'full': len(sward), 'excluded': len(sward_withheld), 'released': len(sward_release)},
#               'source_switches': EXCLUDE_SOURCES}

#     (REPORTS_ROOT / 'release_summary.json').write_text(json.dumps(totals, indent = 2), encoding='utf-8')

#     print('Public datasets saved to:', *map(str, targets), sep = '\n- ')
#     print('Reports saved to:', str(REPORTS_ROOT / 'source_release_counts.csv'), str(REPORTS_ROOT / 'coest_subproject_counts.csv'), str(REPORTS_ROOT / 'release_summary.json'), sep = '\n- ')

In [10]:
# Uncomment only after reviewing the displayed counts and license headings.
# write_release()